# Chapter 5 — Analog Circuit Design Fundamentals for EDA

**Multi-Agent Analog EDA — PhD-Level Monograph (Notebook Form)**

---

Analog EDA agents must reason about **continuous physics**, **stochastic process variation**, and **multi-objective tradeoffs** that have no crisp Boolean truth values.
This chapter supplies the **circuit-theoretic substrate** that makes tool outputs (SPICE, DRC, LVS) *semantically legible* to learning systems: from netlist grammar to small-signal poles, from SKY130 device physics to Pareto-optimal amplifier sizing.

### Learning objectives

1. **Contrast** analog vs digital abstraction: continuous-valued optimization, noise, and margin concepts.
2. **Model** core primitives (mirrors, differential pairs, OTAs) with **equations + schematic box diagrams**.
3. **Parse and generate** SPICE netlists; relate **DC / AC / TRAN / NOISE** to design questions.
4. **Explain** PDK contents and **SKY130**-style parameters; apply **$g_m/I_D$** sizing methodology.
5. **Quantify** GBW, phase margin, CMRR, PSRR, noise; visualize **interactive Pareto** tradeoffs.
6. **Derive** half-circuit symmetry, **Miller** compensation, and **gain–bandwidth** scaling.

### Notation

- Small-signal voltages/currents: $v$, $i$; DC bias: uppercase $V$, $I$.
- Transconductance $g_m$, output conductance $g_{ds}$, capacitances $C_{gs}, C_{gd}, C_L$.
- Laplace variable $s=\sigma+j\omega$; $j=\sqrt{-1}$.

---


## 5.1 Continuous vs. Discrete Signals — Why Analog EDA Is Intrinsically Different

**Digital** design optimizes over a *finite* alphabet of logic levels guarded by **noise margins** $NM_H$, $NM_L$. Once timing closure and Boolean correctness hold, many layers collapse to graph problems on discrete nets.

**Analog** signals live in $\mathbb{R}^n$ (voltages, currents, charges). There is no finite alphabet: **signal integrity** is the study of how noise, distortion, and coupling map a *continuous* intended waveform into a received random process.
Optimization is therefore **continuous-valued** and often **non-convex** (e.g., multiple local minima in phase margin subject to slew constraints).

Let $x(t)$ be a nominal signal and $n(t)$ a zero-mean noise with PSD $S_n(f)$. A useful scalar figure is the **SNR** after integration in bandwidth $B$:
$$
\mathrm{SNR} = \frac{P_{\mathrm{sig}}}{\int_{0}^{B} S_n(f)\,df}.
$$

**Noise margin** in analog is replaced by **dynamic range**, **headroom** to supply rails, and **linear range** of devices (e.g., differential pair $\approx \sqrt{2}V_{\mathrm{ov}}$ in strong inversion, approximately).

**Implication for agents:** verification is rarely a single SAT query; it is a **functional + statistical** certificate: corners, Monte Carlo yield, and worst-case eye diagrams.

---


In [ ]:
from __future__ import annotations

import sys; sys.path.insert(0, '..')
from style_utils import (setup_3b1b_style, glow_line, glow_fill, styled_box,
                         styled_arrow, finish_plot, plotly_3b1b_layout,
                         BACKGROUND, SURFACE, TEXT, TEXT_DIM, GRID,
                         BLUE, TEAL, GREEN, YELLOW, GOLD, RED,
                         ROSE, PURPLE, CYAN, ORANGE, PALETTE)
setup_3b1b_style()

# Imports, dark themes, helpers (self-contained)

import re
import textwrap
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import plotly.graph_objects as go
import plotly.io as pio

DARK_BG = "#0d1117"
ACCENT = "#58a6ff"
GREEN = "#3fb950"
ORANGE = "#d29922"
RED = "#f85149"
VIOLET = "#a371f7"

def apply_mpl_dark():
    mpl.rcParams.update({
        "figure.facecolor": DARK_BG,
        "axes.facecolor": DARK_BG,
        "axes.edgecolor": "#30363d",
        "axes.labelcolor": "#c9d1d9",
        "text.color": "#c9d1d9",
        "xtick.color": "#8b949e",
        "ytick.color": "#8b949e",
        "grid.color": "#21262d",
        "grid.alpha": 0.65,
        "legend.facecolor": "#161b22",
        "legend.edgecolor": "#30363d",
        "font.size": 11,
    })

apply_mpl_dark()
pio.templates.default = "plotly_dark"

RNG = np.random.default_rng(2025)

def finalize_fig(fig):
    fig.patch.set_facecolor(DARK_BG)
    fig.tight_layout()
    return fig

print("Ready: numpy, matplotlib (#0d1117), plotly_dark")

In [ ]:
# Fig 5.1 — Idealized digital levels vs continuous analog waveform + quantization error
apply_mpl_dark()
T = np.linspace(0, 4e-9, 2000)
f0 = 1e9
# Bandlimited analog tone
x_analog = 0.35 * np.sin(2 * np.pi * f0 * T) + 0.05 * np.sin(2 * np.pi * 3 * f0 * T)
# Additive thermal noise (discrete-time approximation)
noise = 0.012 * RNG.standard_normal(size=T.shape)
y = x_analog + noise

# 3-bit quantizer (digital abstraction)
nbits = 3
levels = np.linspace(-0.5, 0.5, 2**nbits)
def quantize(v, levels):
    idx = np.argmin(np.abs(levels[:, None] - v[None, :]), axis=0)
    return levels[idx]

yq = quantize(y, levels)
qerr = y - yq

fig, ax = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True)
ax[0].plot(T * 1e9, y, color=ACCENT, lw=1.0, label="continuous + noise")
ax[0].plot(T * 1e9, yq, color=GREEN, lw=1.2, drawstyle="steps-post", label="3-bit digital levels")
ax[0].set_ylabel("amplitude (a.u.)")
ax[0].set_title("Continuous analog vs quantized digital abstraction")
ax[0].legend(loc="upper right", framealpha=0.9)
ax[0].grid(True)

ax[1].plot(T * 1e9, qerr, color=ORANGE, lw=0.9, label="quantization + noise residual")
ax[1].set_xlabel("time (ns)")
ax[1].set_ylabel("error")
ax[1].legend(loc="upper right", framealpha=0.9)
ax[1].grid(True)

finalize_fig(fig)
plt.show()

snr_est = 10 * np.log10(np.var(x_analog) / np.var(noise))
print(f"Empirical SNR (signal vs additive noise): {snr_est:.1f} dB")


## 5.2 Analog Primitives — Mirrors, Differential Pairs, OTAs

### 5.2.1 Current mirrors

A **simple NMOS mirror** copies $I_{\mathrm{REF}}$ to $I_{\mathrm{OUT}}$ if both devices share the same $V_{GS}$ and remain in saturation:
$$
I_{\mathrm{OUT}} = \frac{(W/L)_2}{(W/L)_1} I_{\mathrm{REF}} \quad \text{(ideal long-channel; short-channel adds } \lambda, V_{DS}\text{ dependence).}
$$

**Output resistance** (small-signal) $r_{o} \approx 1/( \lambda I_D)$ raises **systematic error** when $V_{DS2} \neq V_{DS1}$.

**Cascode mirror** stacks a **cascode device** to shield the copy transistor from $V_{DS}$ swing, boosting output resistance $\approx g_m r_o^2$ (order-of-magnitude).

**Wide-swing / low-voltage** variants bias the cascode gate so all devices remain saturated with minimal headroom (e.g., $V_{DS,\min} \approx 2V_{\mathrm{ov}}$ architectures in classical texts).

### 5.2.2 Differential pair

Tail current $I_{SS}$ splits between two matched transistors. In **strong inversion** square-law region, differential drain current:
$$
\Delta I_d = I_{d1}-I_{d2} = \frac{1}{2}\mu_n C_{ox}\frac{W}{L} V_{id}\sqrt{\frac{4I_{SS}}{\mu_n C_{ox}\frac{W}{L}}-V_{id}^2}, \quad |V_{id}|\le V_{id,\max}.
$$

Small-signal transconductance of the pair (each side) $g_{m1}=g_{m2}=g_m$. **Common-mode gain** $A_{cm}$ arises from finite output conductance of the tail and **mismatch**; **differential gain** $A_{dm}=-g_m R_L$ (resistive load) or $-g_m r_o$ (active load).

**CMRR** (common-mode rejection ratio):
$$
\mathrm{CMRR} = \left|\frac{A_{dm}}{A_{cm}}\right|, \quad \text{often expressed in dB: } 20\log_{10}\mathrm{CMRR}.
$$

### 5.2.3 OTAs

- **Single-stage** (telescopic / folded-cascode): one dominant high-impedance node → **one pole** set by $R_{\mathrm{out}}C_L$.
- **Two-stage Miller**: poles at input of second stage and output; **Miller capacitor** $C_m$ splits poles via **pole splitting** but introduces **RHP zero** unless nulled (e.g., series resistor or cascode).

---


In [ ]:
# Matplotlib "box" schematics: simple mirror, cascode mirror, diff pair, two-stage Miller OTA

def draw_box(ax, xy, w, h, label, fc="#161b22", ec=ACCENT):
    x, y = xy
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.05",
                           linewidth=1.4, edgecolor=ec, facecolor=fc)
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=9, color="#c9d1d9")

def draw_arrow(ax, p0, p1, text=None, color="#8b949e"):
    arr = FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=12, linewidth=1.1, color=color)
    ax.add_patch(arr)
    if text:
        mx, my = (p0[0]+p1[0])/2, (p0[1]+p1[1])/2
        ax.text(mx, my + 0.08, text, ha="center", fontsize=8, color="#8b949e")

def schematic_simple_mirror():
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 3)
    ax.axis("off")
    draw_box(ax, (0.4, 1.0), 1.6, 1.0, "M1\n(W/L)₁")
    draw_box(ax, (3.0, 1.0), 1.6, 1.0, "M2\n(W/L)₂")
    draw_arrow(ax, (2.0, 1.5), (3.0, 1.5), "VGS")
    ax.plot([2.0, 2.0], [1.5, 2.6], color=GREEN, lw=1.2)
    ax.plot([2.0, 4.6], [2.6, 2.6], color=GREEN, lw=1.2)
    ax.text(2.0, 2.75, "gate tie", ha="center", fontsize=8, color=GREEN)
    ax.text(0.2, 0.55, "IREF", color=ORANGE, fontsize=9)
    ax.text(4.8, 0.55, "IOUT", color=ORANGE, fontsize=9)
    ax.text(5.2, 1.35, r"$I_{OUT} \approx \frac{(W/L)_2}{(W/L)_1} I_{REF}$", fontsize=10, color="#c9d1d9")
    ax.set_title("Simple NMOS current mirror (conceptual)")
    finalize_fig(fig)
    plt.show()

def schematic_cascode_mirror():
    fig, ax = plt.subplots(figsize=(9, 4.0))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 4)
    ax.axis("off")
    draw_box(ax, (0.3, 2.0), 1.4, 0.85, "M1")
    draw_box(ax, (0.3, 0.7), 1.4, 0.85, "M3\nCascode")
    draw_box(ax, (2.8, 2.0), 1.4, 0.85, "M2")
    draw_box(ax, (2.8, 0.7), 1.4, 0.85, "M4\nCascode")
    draw_arrow(ax, (1.7, 2.42), (2.8, 2.42))
    ax.plot([1.7, 1.7], [2.42, 3.3], color=GREEN, lw=1.1)
    ax.plot([1.7, 4.2], [3.3, 3.3], color=GREEN, lw=1.1)
    ax.text(4.4, 3.15, "biased cascode gates", fontsize=8, color=GREEN)
    ax.text(5.0, 1.6, r"$R_{out} \uparrow$ (shield $V_{DS}$)", fontsize=10)
    ax.set_title("Cascode current mirror (stacked outputs)")
    finalize_fig(fig)
    plt.show()

def schematic_diffpair():
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 4.2)
    ax.axis("off")
    draw_box(ax, (2.0, 2.3), 1.5, 0.9, "M1")
    draw_box(ax, (4.2, 2.3), 1.5, 0.9, "M2")
    draw_box(ax, (3.1, 0.5), 1.5, 0.9, "ISS\nTail")
    draw_arrow(ax, (1.0, 2.75), (2.0, 2.75), "Vin+")
    draw_arrow(ax, (6.7, 2.75), (5.7, 2.75), "Vin-")
    ax.plot([3.35, 3.35], [1.4, 2.3], color=VIOLET, lw=1.2)
    ax.plot([4.85, 4.85], [1.4, 2.3], color=VIOLET, lw=1.2)
    ax.plot([3.35, 4.85], [1.4, 1.4], color=VIOLET, lw=1.2)
    ax.plot([4.1, 4.1], [0.5, 1.4], color=VIOLET, lw=1.2)
    ax.text(6.8, 2.5, r"$A_{dm} \approx -g_m R_L$", fontsize=10)
    ax.text(6.8, 1.9, r"$A_{cm}$ via $g_{ds}$, mismatch", fontsize=9, color="#8b949e")
    ax.set_title("MOS differential pair + tail current")
    finalize_fig(fig)
    plt.show()

def schematic_two_stage():
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.set_xlim(0, 11)
    ax.set_ylim(0, 3.5)
    ax.axis("off")
    draw_box(ax, (0.4, 1.0), 1.8, 1.1, "Diffpair\n+ loads", ec=GREEN)
    draw_box(ax, (3.0, 1.0), 1.8, 1.1, "G_m1", ec=ACCENT)
    draw_box(ax, (5.6, 1.0), 1.8, 1.1, "G_m2\noutput", ec=ORANGE)
    draw_arrow(ax, (2.2, 1.55), (3.0, 1.55))
    draw_arrow(ax, (4.8, 1.55), (5.6, 1.55))
    # Miller cap feedback
    ax.annotate("", xy=(5.6, 2.35), xytext=(7.4, 2.35),
                arrowprops=dict(arrowstyle="-|>", color=VIOLET, lw=1.2))
    ax.annotate("", xy=(7.4, 1.15), xytext=(5.6, 1.15),
                arrowprops=dict(arrowstyle="-|>", color=VIOLET, lw=1.2))
    ax.plot([7.4, 7.4], [1.15, 2.35], color=VIOLET, lw=1.2)
    ax.text(7.55, 1.7, r"$C_m$", color=VIOLET, fontsize=11)
    ax.text(8.0, 2.5, "Miller: splits poles\nintroduces LHP/RHP zero", fontsize=8, color="#8b949e")
    ax.set_title("Two-stage OTA with Miller compensation (block diagram)")
    finalize_fig(fig)
    plt.show()

schematic_simple_mirror()
schematic_cascode_mirror()
schematic_diffpair()
schematic_two_stage()


### 5.2.4 OTA topologies in practice (single-stage vs folded cascode)

**Telescopic cascode OTA** stacks NMOS (or PMOS) devices to obtain $G_m \approx g_{m1}$ and $R_{\mathrm{out}}$ on the order of $(g_m r_o^2)$-type products, but **output swing** is limited because each stacked device requires $V_{\mathrm{DS,sat}}$ headroom.

**Folded cascode** steers the small-signal current into a **complementary** stack (e.g., NMOS input pair feeding PMOS cascodes), trading **extra bias current**, **more internal nodes** (fold pole), and **layout complexity** for **wider swing** and flexible biasing toward the rails.

**Two-stage Miller** achieves large DC gain without deep vertical stacking at a single output, at the cost of **compensation** (Miller $C_m$, possible nulling resistor, attention to RHP zeros).

---


In [ ]:
# Folded-cascode OTA — block-level signal path (matplotlib)

apply_mpl_dark()

fig, ax = plt.subplots(figsize=(10.5, 4.2))
ax.set_xlim(0, 11.5)
ax.set_ylim(0, 4.0)
ax.axis("off")


def draw_box_fc(ax, xy, w, h, label, ec=ACCENT):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.02,rounding_size=0.05",
        linewidth=1.3,
        edgecolor=ec,
        facecolor="#161b22",
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=8.5, color="#c9d1d9")


def arr_fc(p0, p1, c="#8b949e"):
    ax.add_patch(
        FancyArrowPatch(
            p0, p1, arrowstyle="-|>", mutation_scale=11, linewidth=1.05, color=c
        )
    )


draw_box_fc(ax, (0.4, 2.2), 1.7, 0.95, "NMOS\nDiffpair", ec=GREEN)
draw_box_fc(ax, (2.6, 2.2), 1.5, 0.95, "NMOS\ncascode", ec=GREEN)
draw_box_fc(ax, (2.6, 0.6), 1.5, 0.95, "PMOS\nfold device", ec=VIOLET)
draw_box_fc(ax, (4.5, 0.6), 1.5, 0.95, "PMOS\ncascode", ec=VIOLET)
draw_box_fc(ax, (6.5, 0.9), 1.7, 1.1, "Output\nhigh-Z node", ec=ORANGE)

arr_fc((2.1, 2.67), (2.6, 2.67))
arr_fc((4.1, 2.67), (4.5, 1.05))
ax.annotate("fold", xy=(4.0, 1.85), fontsize=8, color="#8b949e")
arr_fc((6.0, 1.05), (6.5, 1.45))
arr_fc((8.2, 1.45), (8.8, 1.45))
ax.text(9.0, 1.55, r"$G_m \approx g_{m,\mathrm{in}}$", fontsize=9, color="#c9d1d9")
ax.text(9.0, 0.95, r"$R_{\mathrm{out}}$ large (cascodes)", fontsize=9, color="#8b949e")
ax.set_title("Folded-cascode OTA — conceptual block diagram")
finalize_fig(fig)
plt.show()


In [ ]:
# Numerical illustrations: mirror mismatch vs VDS; diff-pair tanh-like transfer (normalized)

apply_mpl_dark()
# Simple mirror systematic error model: Iout = Iref * ratio * (1 + lambda * deltaVDS)
lam = 0.08  # 1/V illustrative
ratio = 2.0
Iref = 50e-6
delta_vds = np.linspace(-0.2, 0.2, 300)
Iout_simple = Iref * ratio * (1 + lam * delta_vds)
# Cascode: suppressed lambda effect (model as smaller effective lambda)
Iout_casc = Iref * ratio * (1 + 0.01 * lam * delta_vds)

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.plot(delta_vds * 1e3, Iout_simple * 1e6, label="simple mirror (illustrative)", color=ACCENT)
ax.plot(delta_vds * 1e3, Iout_casc * 1e6, label="cascode (suppressed ΔVDS)", color=GREEN)
ax.set_xlabel("ΔVDS between mirror legs (mV)")
ax.set_ylabel("Iout (µA)")
ax.set_title("Mirror output current sensitivity (toy λ model)")
ax.legend()
ax.grid(True)
finalize_fig(fig)
plt.show()

# Normalized differential pair drain current difference (long-channel square-law)
# id = 0.5 * beta * (vgs - vt)^2; use normalized Vid / Vov_max
u = np.linspace(-1.2, 1.2, 400)
# i_diff / Iss from textbook strong-inversion expression (normalized)
inside = np.maximum(0.0, 4 - u**2)
i_norm = 0.5 * u * np.sqrt(inside)

fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.plot(u, i_norm, color=VIOLET, lw=1.5)
ax2.set_xlabel(r"normalized $V_{id} / V_{ov,ss}$ (illustrative scale)")
ax2.set_ylabel(r"normalized $(I_{d1}-I_{d2})/I_{SS}$")
ax2.set_title("Differential pair large-signal transconductance characteristic (ideal square-law)")
ax2.grid(True)
finalize_fig(fig2)
plt.show()


## 5.3 SPICE Simulation Fundamentals

**SPICE** (Simulation Program with Integrated Circuit Emphasis) solves nonlinear differential-algebraic equations (DAEs) from device models + KCL/KVL.

### Netlist grammar (typical)

- **Elements**: `R`, `C`, `L`, `M` (MOSFET), `V`, `I`, controlled sources.
- **Nodes**: alphanumeric labels; `0` is global reference.
- **Models**: `.model` or `.lib` includes (foundry decks).

### Analyses

| Card | Question answered |
|------|-------------------|
| `.op` / `.dc` | Bias point, sweeps |
| `.ac` | Small-signal frequency response |
| `.tran` | Time-domain waveforms |
| `.noise` | Integrated / spot noise contributors |

### Corner analysis

Process corners encode **fast/slow** NMOS/PMOS combinations: **TT** (typical-typical), **FF**, **SS**, **FS**, **SF**. Each corner perturbs $V_{th}$, $\mu$, $t_{ox}$, etc., according to the PDK.

### Monte Carlo

Parameters become random variables $\theta\sim p(\theta\mid\text{process})$. **Yield** is
$$
Y = \mathbb{P}[\text{all specs satisfied}] = \int \mathbf{1}\{\phi(x;\theta)\le 0\}\, p(\theta)\,d\theta,
$$
estimated by sample averages.

**Agent affordance:** netlists are *structured text*; parsing yields a graph amenable to constraint checking before invoking the simulator.

---


In [ ]:
# Example SPICE netlists as strings + lightweight parser (educational, not a full SPICE front-end)

@dataclass
class Mosfet:
    name: str
    d: str
    g: str
    s: str
    b: str
    model: str
    w: float
    l: float

@dataclass
class VoltageSource:
    name: str
    nplus: str
    nminus: str
    dc: Optional[float] = None
    ac_mag: Optional[float] = None

@dataclass
class ParsedNetlist:
    title: str = ""
    mosfets: List[Mosfet] = field(default_factory=list)
    voltages: List[VoltageSource] = field(default_factory=list)
    analyses: List[str] = field(default_factory=list)
    raw_lines: List[str] = field(default_factory=list)


def parse_spice_netlist(text: str) -> ParsedNetlist:
    nl = ParsedNetlist()
    lines = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("*"):
            continue
        if line.startswith("."):
            if line.lower().startswith(".title"):
                nl.title = line.split(None, 1)[1] if len(line.split()) > 1 else ""
            else:
                nl.analyses.append(line)
            continue
        lines.append(line)
    nl.raw_lines = lines

    tok_line = re.compile(
        r"^(?P<name>M\w+)\s+(?P<d>\w+)\s+(?P<g>\w+)\s+(?P<s>\w+)\s+(?P<b>\w+)\s+(?P<model>\w+)\s+W=(?P<w>[\d.eE+-]+)\s+L=(?P<l>[\d.eE+-]+)",
        re.I,
    )
    v_line = re.compile(
        r"^(?P<name>V\w+)\s+(?P<p>\w+)\s+(?P<m>\w+)\s+DC\s+(?P<dc>[\d.eE+-]+)(?:\s+AC\s+(?P<ac>[\d.eE+-]+))?",
        re.I,
    )

    for ln in lines:
        m = tok_line.match(ln)
        if m:
            nl.mosfets.append(
                Mosfet(
                    m.group("name"),
                    m.group("d"),
                    m.group("g"),
                    m.group("s"),
                    m.group("b"),
                    m.group("model"),
                    float(m.group("w")),
                    float(m.group("l")),
                )
            )
            continue
        vm = v_line.match(ln)
        if vm:
            nl.voltages.append(
                VoltageSource(
                    vm.group("name"),
                    vm.group("p"),
                    vm.group("m"),
                    dc=float(vm.group("dc")),
                    ac_mag=float(vm.group("ac")) if vm.group("ac") else None,
                )
            )
    return nl


common_footer = textwrap.dedent("""
    * Analyses
    .op
    .ac dec 50 1 1e9
    .tran 1n 200n
    .noise v(out) VIN 1
    .end
""")

netlist_cs_amp = textwrap.dedent("""
    * Simple common-source stage (illustrative element-level)
    .title CS_Stage_Tutorial
    VDD vdd 0 DC 1.8
    VIN in 0 DC 0.65 AC 1
    RD vdd out 50k
    M1 out in 0 0 nch W=2.0u L=0.15u
    .model nch nmos (LEVEL=1 VTO=0.4 KP=200u LAMBDA=0.05)
""") + common_footer

nl = parse_spice_netlist(netlist_cs_amp)
print("TITLE:", nl.title)
print("MOSFETs:", [(m.name, m.model, f"W={m.w}, L={m.l}") for m in nl.mosfets])
print("VSRC:", [(v.name, v.dc, v.ac_mag) for v in nl.voltages])
print("ANALYSES (first 4):", nl.analyses[:4])

# Corner deck fragment as strings
corners = {
    "TT": {"VTO_n": 0.40, "VTO_p": -0.40, "KP_n": 200e-6, "KP_p": 50e-6},
    "FF": {"VTO_n": 0.35, "VTO_p": -0.45, "KP_n": 230e-6, "KP_p": 58e-6},
    "SS": {"VTO_n": 0.45, "VTO_p": -0.35, "KP_n": 170e-6, "KP_p": 42e-6},
    "FS": {"VTO_n": 0.35, "VTO_p": -0.35, "KP_n": 230e-6, "KP_p": 42e-6},
    "SF": {"VTO_n": 0.45, "VTO_p": -0.45, "KP_n": 170e-6, "KP_p": 58e-6},
}
print("Example corner keys:", list(corners.keys()))


In [ ]:
# Monte Carlo toy: GBW vs random Vth mismatch for a single-pole OTA model

apply_mpl_dark()

def gbw_single_pole(gm: float, CL: float) -> float:
    return gm / (2 * np.pi * CL)

N = 2000
gm0 = 1.2e-3
CL = 2e-12
# relative gm variation (process + mismatch abstracted)
sigma_gm = 0.06
gm_s = gm0 * (1 + RNG.normal(0, sigma_gm, size=N))
gbw = gbw_single_pole(gm_s, CL)

# spec: GBW > 85 MHz
spec = 85e6
yield_est = np.mean(gbw > spec)

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.hist(gbw / 1e6, bins=40, color=ACCENT, alpha=0.85, edgecolor=DARK_BG)
ax.axvline(spec / 1e6, color=RED, lw=2, label=f"spec = {spec/1e6:.0f} MHz")
ax.set_xlabel("GBW (MHz) — toy Monte Carlo")
ax.set_ylabel("count")
ax.set_title(f"Yield estimate P(GBW > spec) ≈ {yield_est:.3f}")
ax.legend()
ax.grid(True)
finalize_fig(fig)
plt.show()


## 5.4 SKY130 PDK Overview (Conceptual + Sizing)

A **Process Design Kit (PDK)** bundles:

1. **Device models** (industry-standard BSIM-family cards for SPICE, plus Monte Carlo and corner statistics).
2. **DRC / LVS / ERC** rule decks that define *manufacturable* layouts.
3. **Primitive layout constructs** (fingers, taps, STI rules) and, where applicable, **digital standard-cell libraries** — SKY130 is the **open SkyWater 130 nm** collaboration (Google + SkyWater).

### SKY130 device parameters (symbols agents should recognize)

Use **foundry-shipped** `.lib` corners for sign-off; the following are **pedagogical** anchors:

- **Threshold voltage** $V_{TH}$ (NMOS $V_{THn}$, PMOS $V_{THp}$): separates subthreshold vs strong inversion in large-signal models; corners shift $V_{TH}$ by $\Delta V_{TH,\sigma}$.
- **Intrinsic transconductance parameter** $\mu_n C_{ox}'$ (NMOS) and $\mu_p C_{ox}'$ (PMOS): appears in strong-inversion square-law as $K_n'=\mu_n C_{ox}'$.
- **Channel-length modulation** $\lambda$ (L-dependent in modern models; $r_o \approx 1/(\lambda I_D)$ in first-order):

$$
I_D = \frac{1}{2}\mu_n C_{ox}'\frac{W}{L}(V_{GS}-V_{TH})^2(1+\lambda V_{DS}) \quad \text{(long-channel mnemonic).}
$$

### Operating regions (NMOS)

**Subthreshold** ($V_{GS}<V_{TH}$, $V_{DS}$ moderately large):

$$
I_D \approx I_0 \exp\!\left(\frac{V_{GS}-V_{TH}}{n V_T}\right)\left(1-\exp\!\left(-\frac{V_{DS}}{V_T}\right)\right), \quad V_T=\frac{kT}{q}.
$$

**Strong inversion** ($V_{GS}-V_{TH}\gg 0$): square-law (above) or empirical $I\text{–}V$ from BSIM.

**Moderate inversion**: neither exponential nor purely square-law; **$g_m/I_D$ lookup** from simulated curves is the robust path.

### $g_m/I_D$ methodology

For a target inversion level, pick $g_m/I_D$; invert PDK-generated curves to obtain $(W/L)$ and $V_{ov}$:

$$
\frac{g_m}{I_D} = f\!\left(\frac{I_D}{W}, L, V_{DS}, V_{SB}\right), \qquad g_m = \frac{\partial I_D}{\partial V_{GS}}.
$$

This is the standard **continuous sizing** interface for optimization and for agents that search over bias space.

---


In [ ]:
# gm/Id lookup surrogate + inversion regime chart (toy smooth model — not foundry data)

apply_mpl_dark()

def gm_over_id_weak_strong(id_w: float) -> float:
    """Toy saturating curve: high at low Id/W (weak), asymptote in strong."""
    # id_w in A/m of effective width
    x = np.log10(np.maximum(id_w, 1e-12))
    # piecewise logistic blend
    s = 1.0 / (1.0 + np.exp(2.0 * (x + 3.8)))  # weak side
    gm_id_weak = 25.0  # V^-1 scale
    gm_id_strong = 8.0
    return gm_id_weak * s + gm_id_strong * (1 - s)

id_w = np.logspace(-5.5, -2.5, 400)  # A/m illustrative
gm_id = np.array([gm_over_id_weak_strong(x) for x in id_w])

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(id_w * 1e6, gm_id, color=GREEN, lw=1.6)
ax.set_xscale("log")
ax.axvspan(1e-3, 5e-3, color=ACCENT, alpha=0.12, label="weak–moderate (illustrative band)")
ax.axvspan(5e-3, 2e-2, color=VIOLET, alpha=0.12, label="strong (illustrative band)")
ax.set_xlabel(r"$I_D/W$ (µA/µm illustrative)")
ax.set_ylabel(r"$g_m/I_D$ (V$^{-1}$)")
ax.set_title("Toy $g_m/I_D$ vs current density (agent-facing intuition)")
ax.legend(loc="upper right")
ax.grid(True, which="both", ls=":", alpha=0.5)
finalize_fig(fig)
plt.show()


## 5.5 Key Performance Metrics and Tradeoffs

For a two-pole dominant OTA with unity-gain frequency $\omega_u$ and second pole $\omega_2$:

**Gain-bandwidth product** (dominant-pole approximation):
$$
\mathrm{GBW} = \frac{1}{2\pi}\omega_u \approx \frac{G_{m1}}{2\pi C_m}.
$$

**Phase margin** (crude two-pole estimate ignoring zeros):
$$
\mathrm{PM} \approx 90^\circ - \arctan\!\frac{\omega_u}{\omega_2}.
$$

**DC gain** (two-stage illustrative):
$$
A_{0} \approx (g_{m1} r_{o1})(g_{m2} r_{o2}).
$$

**CMRR** limited by tail impedance $R_{SS}$ and asymmetry; **PSRR** splits into **gated** paths from $V_{DD}$ / ground.

**Input-referred thermal noise** (square-law MOS, strong inversion order-of-magnitude):
$$
\overline{v_{n}^2} \approx \frac{8kT\gamma}{g_m} \quad \text{(per effective transistor — see Gray/Hurst/Lewis for full topology sums).}
$$

**Pareto frontier:** maximize GBW and minimize power $P$ and noise $N$ simultaneously is generally **impossible**; the **non-dominated** set illustrates tradeoffs agents must navigate.

---


In [ ]:
# Interactive Pareto frontier: GBW vs Power vs integrated input noise (toy multi-objective)

def ota_toy_metrics(W1_um: float, Itail_uA: float) -> Tuple[float, float, float]:
    """Return GBW (MHz), Power (mW), input noise (µV rms) — illustrative power-law scalings."""
    Itail = Itail_uA * 1e-6
    # gm ~ sqrt(2 mu Cox W/L Id) style -> scale sqrt(W * I)
    gm = 1e-3 * np.sqrt(W1_um * Itail_uA / 100.0)
    CL = 1.5e-12
    gbw = gm / (2 * np.pi * CL) / 1e6
    Vdd = 1.8
    P = Vdd * Itail_uA * 1e-3 * 1.1  # extra bias overhead
    # noise ~ kT/gm
    vn = np.sqrt(8 * 1.38e-23 * 300 / gm) * 1e6
    return gbw, P, vn

Wgrid = np.linspace(5, 80, 35)
Igrid = np.linspace(20, 250, 35)
pts = []
for w in Wgrid:
    for i in Igrid:
        gbw, p, vn = ota_toy_metrics(w, i)
        pts.append((gbw, p, vn, w, i))
pts = np.array(pts)

# Non-dominated w.r.t. maximize GBW, minimize P, minimize vn
def pareto_mask(PA):
    """Return True for non-dominated points: maximize gbw, minimize power, minimize noise."""
    gbw, pwr, noise = PA[:, 0], PA[:, 1], PA[:, 2]
    n = len(PA)
    m = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if (
                gbw[j] >= gbw[i]
                and pwr[j] <= pwr[i]
                and noise[j] <= noise[i]
                and (
                    gbw[j] > gbw[i]
                    or pwr[j] < pwr[i]
                    or noise[j] < noise[i]
                )
            ):
                m[i] = False
                break
    return m

mask = pareto_mask(pts[:, :3])
pareto = pts[mask]

fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=pts[:, 0],
        y=pts[:, 1],
        z=pts[:, 2],
        mode="markers",
        marker=dict(size=2, color="#484f58", opacity=0.35),
        name="grid samples",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=pareto[:, 0],
        y=pareto[:, 1],
        z=pareto[:, 2],
        mode="markers",
        marker=dict(size=4, color="#58a6ff"),
        name="Pareto frontier (non-dominated)",
    )
)
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117",
    title="Toy Pareto surface: GBW ↑, Power ↓, Input noise ↓",
    scene=dict(
        xaxis_title="GBW (MHz)",
        yaxis_title="Power (mW)",
        zaxis_title="Input noise (µV rms)",
        bgcolor="#0d1117",
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    height=520,
)
fig.show()


## 5.6 Small-Signal Analysis — Half-Circuit, Miller, GBW Derivation

### Half-circuit (symmetric differential pair)

For purely **differential** excitation about a symmetric bias, each half-circuit sees **half** the differential voltage at the gate and the **virtual ground** at the tail node (for ideal tail current). The **common-mode** half-circuit doubles the tail impedance effect.

### Miller effect

Impedance $Z$ bridging input and output of a gain block $A_v$ is seen at the input approximately as
$$
Z_{\mathrm{in,Miller}} \approx \frac{Z}{1-A_v}.
$$
For a capacitor $C_f$, **input capacitance** inflates by $(1-A_v)$, **output** by $(1-1/A_v)\approx 1$ for large $|A_v|$.

### Two-stage GBW with dominant Miller pole

Let $G_{m1}$ be first-stage transconductance, $R_1$ the first-stage output resistance, $G_{m2}$ the second stage, $R_2$ the output resistance, load $C_L$, Miller cap $C_m$. A standard dominant-pole placement sets
$$
p_1 \approx -\frac{1}{R_1\big(C_{m}(1+|A_2|)\big)}, \quad A_2 = -G_{m2}R_2.
$$
Unity-gain frequency $\omega_u \approx G_{m1}/C_m$ when the second pole $\omega_2$ is pushed beyond $\omega_u$ via **pole splitting**. Increasing $C_m$ lowers $\omega_u$ (slower) but **improves** phase margin by reducing $\omega_u/\omega_2$.

---


In [ ]:
# Miller capacitor effective input capacitance + Bode-style magnitude sketch

apply_mpl_dark()

Av2 = -48.0  # second stage DC gain magnitude (negative inverting)
Cf = 50e-15
C_pi = 20e-15

Cmiller = Cf * (1 + abs(Av2))
Cin_eff = C_pi + Cmiller

f = np.logspace(6, 10, 500)
omega = 2 * np.pi * f
# Toy two-pole open-loop gain: A0 / ((1+j w/w1)(1+j w/w2))
A0 = 2000.0
w1 = 2 * np.pi * 5e3
w2 = 2 * np.pi * 120e6
H = A0 / ((1 + 1j * omega / w1) * (1 + 1j * omega / w2))
mag = 20 * np.log10(np.maximum(np.abs(H), 1e-12))

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.semilogx(f / 1e6, mag, color=ACCENT, lw=1.4)
ax.axhline(0, color=GREEN, ls="--", lw=1.0, label="0 dB (unity loop gain ref)")
ax.set_xlabel("frequency (MHz)")
ax.set_ylabel("open-loop gain (dB) — illustrative")
ax.set_title(f"Miller splits poles: effective input cap += Cf×(1+|Av2|) = {Cmiller*1e15:.1f} fF (example)")
ax.grid(True, which="both", ls=":", alpha=0.55)
ax.legend()
finalize_fig(fig)
plt.show()

# Phase margin estimate from unity crossing
idx0 = np.where(mag <= 0)[0]
if len(idx0):
    i0 = idx0[0]
    phase = np.angle(H[i0], deg=True)
    pm = 180 + phase  # for inverting dominant assumption
    print(f"Unity-gain frequency ≈ {f[i0]/1e6:.2f} MHz; phase ≈ {phase:.1f}°; PM ≈ {pm:.1f}°")

print(f"Cπ = {C_pi*1e15:.0f} fF, Cf = {Cf*1e15:.0f} fF, |Av2| = {abs(Av2):.1f}")


## 5.7 Chapter synthesis — What an analog EDA agent should internalize

1. **Continuous state** ⇒ optimization lives in $\mathbb{R}^n$; noise and variation are first-class.
2. **Primitives** (mirror, diffpair, OTA) are the **grammar** of schematics; each has a small-signal story.
3. **SPICE** is the **executable semantics** of a netlist; parsing + static checks reduce tool-call failures.
4. **PDK + $g_m/I_D$** connect physics to **sizing knobs** with smooth lookup surfaces.
5. **Metrics** form a **multi-objective** landscape; Pareto sets discipline exploration.
6. **Half-circuit + Miller** explain 80% of intuitive pole-zero reasoning in OTAs.

---
